# Reduksi Model Semantik cc.id.300 → cc.id.100 (Kaggle, dari file upload)
**Versi ini memakai file `cc.id.300.bin.gz` yang sudah kamu unduh manual & upload sebagai Kaggle Dataset**
(karena unduh otomatis gagal terus).

**Prasyarat — lakukan SEBELUM menjalankan notebook:**
1. Kaggle → **Datasets → New Dataset** → upload `cc.id.300.bin.gz` → simpan (butuh waktu, sekali saja).
2. Buka notebook ini → panel kanan **Add Data → Your Datasets → pilih dataset tadi**.
3. Settings → **Accelerator: None**, Internet boleh OFF (tak perlu unduh apa pun lagi).
4. Jalankan semua sel.

**Hasil:** `cc.id.100.bin` (~1GB) di `/kaggle/working/` → unduh dari panel Output →
unggah ke Google Drive `MyDrive/skripsi_merek/` → dipakai di Colab Langkah C–G.

> RAM Kaggle (~32GB) cukup untuk memuat model penuh + reduksi. Jalankan sekali saja.


## 1. Cari file .gz otomatis di /kaggle/input (tak perlu tahu path persis)

In [ ]:
import os, glob, time, shutil

# cari semua file cc.id.300 (.gz atau .bin) di seluruh dataset yang di-attach
kandidat = []
for pola in ["**/cc.id.300.bin.gz", "**/cc.id.300.bin", "**/*.gz", "**/*.bin"]:
    kandidat += glob.glob(f"/kaggle/input/{pola}", recursive=True)
kandidat = sorted(set(kandidat))

print("File ditemukan di /kaggle/input:")
for k in kandidat:
    print(f"  {k}  ({os.path.getsize(k)/1e9:.2f} GB)")

# pilih yang paling cocok
src = None
for k in kandidat:
    if k.endswith("cc.id.300.bin.gz"): src = k; break
if src is None:
    for k in kandidat:
        if k.endswith("cc.id.300.bin"): src = k; break
if src is None and kandidat:
    src = kandidat[0]   # fallback: file terbesar/pertama

assert src is not None, "Tidak ada file model di /kaggle/input. Pastikan dataset sudah di-Add Data."
print("\nDipakai:", src)

## 2. Siapkan file di working (ekstrak bila .gz)
`/kaggle/input` bersifat **read-only**, jadi hasil ekstrak ditaruh di `/kaggle/working`.


In [ ]:
os.chdir("/kaggle/working")
BIN = "/kaggle/working/cc.id.300.bin"

t0 = time.time()
if os.path.exists(BIN):
    print("cc.id.300.bin sudah ada di working, lewati ekstrak.")
elif src.endswith(".gz"):
    print("Mengekstrak .gz -> /kaggle/working/cc.id.300.bin ...")
    # gunzip tak bisa nulis ke input (read-only), jadi salurkan ke working
    !gunzip -c "{src}" > "{BIN}"
    print(f"Selesai ekstrak dalam {time.time()-t0:.0f} detik.")
else:
    # sudah .bin: salin ke working agar konsisten
    print("Menyalin .bin ke working ...")
    shutil.copy(src, BIN)

print("Ukuran cc.id.300.bin:", f"{os.path.getsize(BIN)/1e9:.2f} GB")

## 3. Instalasi fasttext

In [ ]:
!pip install fasttext-wheel -q
import fasttext, fasttext.util, numpy as np
print("fasttext siap.")

## 4. Muat model penuh (butuh RAM besar — inilah alasan pakai Kaggle)

In [ ]:
t0 = time.time()
print("Memuat cc.id.300.bin ...")
model = fasttext.load_model(BIN)
print(f"Dimuat dalam {time.time()-t0:.0f} detik. Dimensi: {model.get_dimension()}")

def cos(a,b):
    a,b=np.array(a),np.array(b); return float(a@b/((np.linalg.norm(a)*np.linalg.norm(b))+1e-9))
print("Uji makna (300-dim):")
print("  mawar vs rose :", round(cos(model.get_word_vector("mawar"), model.get_word_vector("rose")),3))
print("  mawar vs meja :", round(cos(model.get_word_vector("mawar"), model.get_word_vector("meja")),3))

## 5. Reduksi 300 → 100 dimensi

In [ ]:
t0 = time.time()
print("Mereduksi 300 -> 100 ...")
fasttext.util.reduce_model(model, 100)
print(f"Selesai dalam {time.time()-t0:.0f} detik. Dimensi sekarang: {model.get_dimension()}")

print("Uji makna (100-dim, setelah reduksi):")
print("  mawar vs rose :", round(cos(model.get_word_vector("mawar"), model.get_word_vector("rose")),3))
print("  mawar vs meja :", round(cos(model.get_word_vector("mawar"), model.get_word_vector("meja")),3))
print("  OOV 'tokopedia':", model.get_word_vector("tokopedia").shape, "(subword tetap jalan)")

## 6. Simpan cc.id.100.bin ke output

In [ ]:
OUT = "/kaggle/working/cc.id.100.bin"
model.save_model(OUT)
print(f"Tersimpan: {OUT} ({os.path.getsize(OUT)/1e9:.2f} GB)")

# bersihkan cc.id.300.bin dari working agar commit output lebih ringan
try:
    os.remove(BIN); print("cc.id.300.bin dihapus dari working.")
except Exception as e:
    print("Lewati pembersihan:", e)

print("\nSELESAI. Langkah berikutnya:")
print("1. Panel 'Output' (kanan) -> unduh cc.id.100.bin")
print("2. Unggah ke Google Drive: MyDrive/skripsi_merek/cc.id.100.bin")
print("3. Colab Langkah C-G memuat file ini dari Drive (path: /content/drive/MyDrive/skripsi_merek/cc.id.100.bin)")

## Catatan
- Jika sel 6 tak memunculkan `cc.id.100.bin` di Output, klik **Save Version → Save & Run All (Commit)**
  agar output ter-commit; file muncul setelah commit selesai.
- Uji "mawar vs rose" tinggi & "mawar vs meja" rendah = reduksi tidak merusak makna. Kalau dua-duanya
  mirip, artinya sinyal semantik lemah (konsisten dgn temuan bahwa nama merek sulit secara semantik).
